# Backhoe and excavator detection — inference

This short notebook loads the released `best.pt` model and predicts three fixed held-out site photographs. It runs on a CPU, without a Roboflow account or other credentials.

## 1. Install the model library

Pin the Ultralytics version used to train the checkpoint.

In [ ]:
%pip install -q "ultralytics==8.4.163"

## 2. Load the frozen model and photos

The version 3 dataset ZIP is checked against its SHA256 fingerprint. Both files are public GitHub Release assets.

In [ ]:
from pathlib import Path
import hashlib
import urllib.request
import zipfile

DATA_URL = "https://github.com/Asiya0555/Backhoe-Excavator-Site-Detection/releases/download/v1.0.0/backhoe-excavator-v3-yolov11.zip"
DATA_SHA256 = "8e3645e885a7f4b3ec90c25cdf39b680a474448659397f123ac13a7d28ccab8a"
MODEL_URL = "https://github.com/Asiya0555/Backhoe-Excavator-Site-Detection/releases/download/v1.0.0/best.pt"
ROOT = Path("/content/backhoe_excavator_data")
ZIP_PATH = Path("/content/backhoe_excavator_v3.zip")
MODEL_PATH = Path("/content/best.pt")

def sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(1 << 20), b""):
            digest.update(block)
    return digest.hexdigest()

if not ZIP_PATH.is_file():
    urllib.request.urlretrieve(DATA_URL, ZIP_PATH)
actual_sha = sha256_file(ZIP_PATH)
if actual_sha != DATA_SHA256:
    ZIP_PATH.unlink(missing_ok=True)
    raise AssertionError(f"Dataset checksum mismatch: {actual_sha}")
print("Dataset SHA256 verified:", actual_sha)

if not list(ROOT.rglob("data.yaml")):
    ROOT.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(ZIP_PATH) as archive:
        archive.extractall(ROOT)

if not MODEL_PATH.is_file():
    urllib.request.urlretrieve(MODEL_URL, MODEL_PATH)
assert MODEL_PATH.stat().st_size > 100_000, "Model checkpoint looks incomplete"

# The export may unpack directly or into a nested project directory.
dataset_roots = [p.parent for p in ROOT.rglob("data.yaml")
                 if (p.parent / "test" / "images").is_dir()]
assert dataset_roots, "Test images not found in the frozen export"
TEST_IMAGES = dataset_roots[0] / "test" / "images"
print("Model ready; test photos:", TEST_IMAGES)

## 3. Predict three held-out photos

These fixed photos demonstrate two backhoe loaders and one excavator. Each displayed image includes the model’s boxes, class labels and confidence scores.

In [ ]:
from ultralytics import YOLO
from PIL import Image
from IPython.display import display

model = YOLO(str(MODEL_PATH))
samples = [
    ("Backhoe loader — rear view", "37874580040398070775d54550f63135", "sample_backhoe_rear.png"),
    ("Backhoe loader — site view", "c2b412fa30a3303832de00b13f24f875", "sample_backhoe_site.png"),
    ("Excavator — site view", "8887f5372e6aa57f98099b2a78dcf2d1", "sample_excavator_site.png"),
]
OUTPUT = Path("/content/prediction_examples")
OUTPUT.mkdir(parents=True, exist_ok=True)

for title, image_id, filename in samples:
    matches = list(TEST_IMAGES.glob(f"*{image_id}*"))
    assert len(matches) == 1, f"Expected one photo for {image_id}, found {len(matches)}"
    result = model.predict(source=str(matches[0]), imgsz=640, conf=0.25,
                           device="cpu", verbose=False)[0]
    visual = Image.fromarray(result.plot()[:, :, ::-1])
    visual.save(OUTPUT / filename)
    labels = [f"{model.names[int(cls)]} {float(score):.2f}"
              for cls, score in zip(result.boxes.cls, result.boxes.conf)]
    print(title, "—", ", ".join(labels) if labels else "none")
    display(visual)
print("Saved annotated examples to:", OUTPUT)

## 4. Try another photo (optional)

Upload a photo using Colab’s file browser and set `CUSTOM_IMAGE` to its path. The default skips this optional step so **Run all** completes without asking for a file.

In [ ]:
CUSTOM_IMAGE = None  # Example: "/content/site_photo.jpg"
if CUSTOM_IMAGE:
    custom_path = Path(CUSTOM_IMAGE)
    assert custom_path.is_file(), f"Photo not found: {custom_path}"
    result = model.predict(source=str(custom_path), imgsz=640, conf=0.25,
                           device="cpu", verbose=False)[0]
    display(Image.fromarray(result.plot()[:, :, ::-1]))
else:
    print("Optional photo skipped. Set CUSTOM_IMAGE to a path to try one.")

## Interpretation

A predicted box is a visual suggestion, not proof of equipment identity. The model has known false positives on lookalike machinery and misses on unusual viewpoints. Review the photo and the [evidence pack](https://github.com/Asiya0555/Backhoe-Excavator-Site-Detection/tree/main/results) before using a result in a project record.